In [46]:
!pip install youtube-transcript-api
!python -m pip install -U langchain-chroma

In [47]:
!python -m pip install -r requirements.txt

In [48]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate
from sentence_transformers import SentenceTransformer


In [49]:
## Document Loader for  a single video
yt_api=YouTubeTranscriptApi()
video_id='4Vz6L8B73i4&t=778s'
video_id = video_id.split("&", 1)[0]
transcripts = yt_api.fetch(video_id)
video_url = f"https://www.youtube.com/watch?v={video_id}"
start_time = 778


In [50]:
transcripts 
for item in transcripts[:5]:
    print(item.text)
    print(item.start)# so we can track the timing of the wods spoken by the speaker
    print()

If you let people have their phones out,
0.08

the IQ of your employees is lower. Cuz
2.159

there was a study done when the phone
5.359

was visible to you. Scores were
8.0

significantly decreased.
10.08



In [51]:
transcripts_text=" ".join(doc.text for doc in transcripts)
metadata = {
    "video_url": video_url
}

In [52]:
transcripts_text

'If you let people have their phones out, the IQ of your employees is lower. Cuz there was a study done when the phone was visible to you. Scores were significantly decreased. >> Like my attention decreased. >> Yes, Raj. It was also general fluid intelligence. That means you\'re dumber. This is the scary reality of how attention works in the brain. people who hold their phone or put their phone out on the table when they\'re talking to people and they first meet you. You seem less trustworthy, less capable, less intelligent, all of these different things. [music] I\'m a cognitive neuroscientist. Study how we can train the brain to be better. Every single time you [music] switch your attention, you pay for it in time and you pay for it in energy. And if you don\'t believe me, we can test it out. >> Grab a piece of paper. You have some paper and a pen. Now, on the top line, [music] I want you to write, "I am a great multitasker." And then on the bottom line, I want you to write the numbe

In [53]:
## Text Splitting 
Splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

In [54]:
chunks = Splitter.split_text(
    transcripts_text
)

In [55]:
len(chunks)

366

In [56]:
pip install -U sentence-transformers

In [57]:
from sentence_transformers import SentenceTransformer
model_name=SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")

Loading weights: 100%|██████████| 310/310 [00:13<00:00, 23.01it/s]


In [58]:
from sentence_transformers import SentenceTransformer
from langchain_chroma import Chroma


# ============================================
# 1. LOAD LOCAL EMBEDDING MODEL
# ============================================

model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B",
    device="cpu"
)

print("Embedding model loaded!")



Loading weights: 100%|██████████| 310/310 [00:09<00:00, 32.76it/s]


Embedding model loaded!


In [59]:
# 2. WRAPPER FOR LANGCHAIN
# ============================================

class SentenceTransformerEmbeddings:

    def __init__(self, model):
        self.model = model

    def embed_documents(self, texts):
        print(f"Embedding {len(texts)} documents...")

        embeddings = self.model.encode(
            texts,
            batch_size=8,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True
        )

        return embeddings.tolist()

    def embed_query(self, text):
        embedding = self.model.encode(
            [text],
            batch_size=1,
            convert_to_numpy=True,
            normalize_embeddings=True
        )

        return embedding[0].tolist()


embeddings = SentenceTransformerEmbeddings(model)

print("Embedding wrapper ready!")
# ============================================
# 3. CHECK YOUR CHUNKS
# ============================================

print("Number of chunks:", len(chunks))

# Example:
# chunks = your Document objects



Embedding wrapper ready!
Number of chunks: 366


In [60]:
metadatas = [
    metadata for _ in chunks
]


In [61]:

# 4. CREATE CHROMA VECTOR STORE


vectorstore = Chroma.from_texts(
    texts=chunks,
    embedding=embeddings,
    metadatas=metadatas,
    collection_name="youtube_rag",
    persist_directory="./chroma_db"
)

print("✅ ChromaDB created successfully!")

Embedding 366 documents...


Batches: 100%|██████████| 46/46 [14:43<00:00, 19.21s/it]


✅ ChromaDB created successfully!


In [62]:
vectorstore.similarity_search(query="how to improve focus ?",k=5)

[Document(id='d23490e8-21f6-4a9d-bd3a-57fb92d65188', metadata={}, page_content='at least introducing a new variable. You\'re at least saying, "Hey, drink some water." At least, seriously, something as small as this. Because anything you do, anything you touch, smell, drink, listen, listen to a different playlist, different music, get up and move to another room. If you\'re sitting at your desk and you\'re always working at your desk and you\'re like, "This is my focus. This is where I\'m at." And you\'re just like, "This is the problem." The problem is also here. It\'s in your'),
 Document(id='d725b5e4-292d-43f2-ba4a-1de302625be9', metadata={}, page_content='at least introducing a new variable. You\'re at least saying, "Hey, drink some water." At least, seriously, something as small as this. Because anything you do, anything you touch, smell, drink, listen, listen to a different playlist, different music, get up and move to another room. If you\'re sitting at your desk and you\'re alwa

In [63]:
retriver=vectorstore.as_retriever(search_kwargs={"k":4})

In [64]:
retriver.invoke("what is Chronotypes?")

[Document(id='d2f27490-1700-4e19-aa5e-b0b7d95248b9', metadata={}, page_content="three different types we talked about AM shifted bifphasic and PM shifted and anybody else who's watching if you also take the assessment you're going to get a report as well a PDF report that's going to go over all of this information so these are the three main chronotypes now as you can see here in the center we have population distributions the population [snorts] distribution here shows that the majority of people in the world are bifphasic >> even they are not like 5 a.m. type >> no"),
 Document(id='cd8a3a2e-23b7-45e9-a15a-252022956a71', metadata={}, page_content="three different types we talked about AM shifted bifphasic and PM shifted and anybody else who's watching if you also take the assessment you're going to get a report as well a PDF report that's going to go over all of this information so these are the three main chronotypes now as you can see here in the center we have population distributi

In [65]:
prompt = PromptTemplate(
    template='''You are a helpful assistant that answers questions
using ONLY information contained in the provided video context.

Use the conversation history only to understand what the user
is referring to.

Rules:

1. Use only information supported by the video context.
2. Do not use outside knowledge.
3. Do not invent facts.
4. Conversation history is only for understanding references
   such as "it", "that", "this", or "the second point".
5. The video context is the only source of factual information.
6. Give a detailed answer when the context contains enough
   information.
7. If the context is insufficient, say:

"I don't know based on the provided context."

Conversation History:
{history}

Video Context:
{context}

Current Question:
{question}

Answer:''',

    input_variables=[
        "history",
        "context",
        "question"
    ]
)


In [66]:
question="Tell me what is behind social media addiction. "

In [67]:
context=retriver.invoke(question)
context=" ".join(i.page_content for i in context)

In [68]:
final_prompt = prompt.invoke({
    "history": history,
    "context": context,
    "question": question
})

In [69]:
LLM=ChatGroq(model="openai/gpt-oss-20b",temperature=0.2)

In [70]:
answer=LLM.invoke(final_prompt)

In [71]:
print(answer.content)

Behind social media addiction, as described in the video, is a cycle of craving for social connection that turns into a quick, pleasurable distraction. People feel “hungry” for the kind of content that their friends would post, but the actual feed is filled with posts from strangers. This mismatch creates a sense of wanting something familiar (the friends’ posts) while actually scrolling through unfamiliar content. The distraction feels good in the moment—much like eating candy when you’re hungry—so the brain rewards the behavior. In short, the addiction is driven by a desire for social connection, the immediate pleasure of scrolling, and the constant, often unfamiliar, supply of content that satisfies that craving.
